In [ ]:
from pathlib import Path
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'common.ipynb').is_file()))
get_ipython().run_line_magic('run', '-i "' + str(ROOT / 'src' / 'common.ipynb') + '"')


In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_percentage_error
FIG = OUT / 'figures'
plt.rcdefaults()
plt.rcParams.update({'svg.fonttype': 'none', 'pdf.fonttype': 42, 'ps.fonttype': 42})


In [ ]:
selected_parameters = select_xgb_parameters()
scores = _XGB_SELECTION["scores"]
summary = _XGB_SELECTION["summary"]
selection_results = _XGB_SELECTION["results"]
display(summary.round(4))


In [ ]:
parameters = dict(selected_parameters, n_jobs=1)
records, dfs_plot = [], {}
for code in MODELS:
    data = load_data(code)
    train, test = split_indices(len(data))
    model = xgb.XGBRegressor(**parameters)
    model.fit(data.iloc[train][FEATURES], data.iloc[train]['EB_pct'])
    part = data.iloc[test]
    prediction = model.predict(part[FEATURES])
    assert (part['EB_pct'] > 0).all()
    dfs_plot[code] = pd.DataFrame({'GEOID': part['GEOID'].to_numpy(), 'observed': part['EB_pct'].to_numpy(), 'predicted': prediction})
    records.append({'model': code, 'n': len(part), 'R2': r2_score(part['EB_pct'], prediction), 'RMSE_pp': root_mean_squared_error(part['EB_pct'], prediction), 'MAPE_pct': 100 * mean_absolute_percentage_error(part['EB_pct'], prediction)})
performance = pd.DataFrame(records)
performance.round(2)


In [ ]:
def density_colors(x, y, bins=70):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    x_edges = np.linspace(np.nanmin(x), np.nanmax(x), bins + 1)
    y_edges = np.linspace(np.nanmin(y), np.nanmax(y), bins + 1)
    H, _, _ = np.histogram2d(x, y, bins=[x_edges, y_edges])
    xi = np.clip(np.digitize(x, x_edges) - 1, 0, bins - 1)
    yi = np.clip(np.digitize(y, y_edges) - 1, 0, bins - 1)
    return np.log1p(H[xi, yi].astype(float))


In [ ]:
def export_figure(fig, stem, performance=False):
    for ax in fig.axes:
        for collection in ax.collections:
            collection.set_rasterized(False)
        if performance or ax.get_xlabel() in ['mean(|SHAP|)', 'SHAP value']:
            ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
        if performance:
            ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
    fig.canvas.draw()
    bounds = {} if stem == 'FigureS3' else {'bbox_inches': 'tight'}
    for ext in ['png', 'pdf', 'svg']:
        fig.savefig(FIG / f'{stem}.{ext}', dpi=300, **bounds)
    plt.close(fig)


In [ ]:
season_order = SEASONS
label_col = 'observed'
TRUE_COL, PRED_COL = ('observed', 'predicted')
all_vals = [a for code in SEASONS for a in [dfs_plot[code]['observed'].to_numpy(), dfs_plot[code]['predicted'].to_numpy()]]


In [ ]:
all_vals = np.concatenate(all_vals)
vmin = np.nanmin(all_vals)
vmax = np.nanmax(all_vals)
pad = 0.03 * (vmax - vmin if vmax > vmin else 1.0)
xmin, xmax = (vmin - pad, vmax + pad)
letters = ['a', 'b', 'c', 'd']
fig, axes = plt.subplots(2, 2, figsize=(7.4, 6.67), sharex=True, sharey=True, constrained_layout=True)
axes = axes.ravel()
for i, (season, ax) in enumerate(zip(season_order, axes)):
    dfp = dfs_plot[season]
    x = dfp[TRUE_COL].values
    y = dfp[PRED_COL].values
    d = density_colors(x, y, bins=70)
    ax.scatter(x, y, c=d, cmap='viridis', s=10, alpha=0.85, linewidths=0)
    ax.plot([xmin, xmax], [xmin, xmax], color='black', lw=1, alpha=0.8)
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(xmin, xmax)
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
    ax.text(-0.1, 1.05, letters[i], transform=ax.transAxes, va='top', ha='left', fontsize=14, fontweight='bold', clip_on=False)
    r2 = r2_score(x, y)
    rmse = root_mean_squared_error(x, y)
    mape = mean_absolute_percentage_error(x, y) * 100
    ax.text(0.02, 0.98, f'$R^2$ = {r2:.2f}\nRMSE = {rmse:.2f}\nMAPE = {mape:.2f}%', transform=ax.transAxes, va='top', ha='left', fontsize=12, bbox=dict(boxstyle='round,pad=0.25', facecolor='white', alpha=0.85, edgecolor='none'))
fig.supxlabel('Monthly Energy Burden (%)', fontsize=13)
fig.supylabel('Predicted Monthly Energy Burden (%)', fontsize=13)
export_figure(fig, 'Figure2', performance=True)


In [ ]:
test_df = dfs_plot['ALL'].rename(columns={'predicted': 'predicted_observed'})
row = performance.set_index('model').loc['ALL']
metrics = {'Test_R2': row['R2'], 'RMSE': row['RMSE_pp'], 'MAPE': row['MAPE_pct'] / 100}


In [ ]:
x = test_df[label_col].values
y = test_df['predicted_' + label_col].values
vmin = np.nanmin([x.min(), y.min()])
vmax = np.nanmax([x.max(), y.max()])
pad = 0.03 * (vmax - vmin if vmax > vmin else 1.0)
xmin, xmax = (vmin - pad, vmax + pad)
d = density_colors(x, y, bins=70)
fig, ax = plt.subplots(1, 1, figsize=(4.2, 4), constrained_layout=True)
ax.scatter(x, y, c=d, cmap='viridis', s=10, alpha=0.85, linewidths=0)
for spine in ax.spines.values():
    spine.set_linewidth(1.5)
ax.plot([xmin, xmax], [xmin, xmax], color='black', lw=1, alpha=0.8)
ax.set_xlim(xmin, xmax)
ax.set_ylim(xmin, xmax)
ax.text(0.02, 0.98, f"$R^2$ = {metrics['Test_R2']:.2f}\nRMSE = {metrics['RMSE']:.2f}\nMAPE = {metrics['MAPE'] * 100:.2f}%", transform=ax.transAxes, va='top', ha='left', fontsize=12, bbox=dict(boxstyle='round,pad=0.25', facecolor='white', alpha=0.85, edgecolor='none'))
ax.set_xlabel('Monthly Energy Burden (%)', fontsize=13)
ax.set_ylabel('Predicted Monthly Energy Burden (%)', fontsize=13)
export_figure(fig, 'FigureS2', performance=True)
